# AISIN IMRA（日本語版・aisin.com）論文一覧スクレイピング

今井先生の元タスク文書「Build a Researcher Database」の3番目の対象のうち、AISIN公式サイト内の日本語ページ。
`https://www.aisin.com/jp/group/imra-japan/paper/page{n}/` の形でページ送りされている（1ページ17件前後×6ページ）。

- 対象URL: `https://www.aisin.com/jp/group/imra-japan/paper/page1/` 〜 `page6/`
- robots.txt確認済み、問題なし
- 構造: `article.item.d_flex` の中に `div.category > span.cat`（カテゴリ）と `div.conts > p.txt`（タイトル・著者・掲載誌がまとめて3行のテキスト）、`a.link`（リンク、多くはdoi.org）
- IMRA/AISIN所属著者の太字表記はない（この一覧内の著者を区別する仕組みは見当たらない）
- 注意: AISIN IMRA関連は`imra.com`（英語・米国）・`imra.eu`（欧州）にも別データがあり、このノートブックはそのうち日本語版1窓口のみを扱う

In [1]:
from datetime import datetime, timezone
from pathlib import Path
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'AGENTS.md').exists() and (candidate / 'data').is_dir():
        ROOT = candidate
        break
else:
    raise RuntimeError('リポジトリ直下を特定できません。リポジトリ内で実行してください。')

OUTPUT_DIR = ROOT / 'data' / 'derived' / 'toyota_official_scrape'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_URL = 'https://www.aisin.com/jp/group/imra-japan/paper/page{n}/'
N_PAGES = 6
USER_AGENT = 'Mozilla/5.0 (compatible; Sci-sci-research-bot/1.0; contact: r.ikura@msp-lab.org)'

RUN_AT_UTC = datetime.now(timezone.utc).isoformat()
print(f'RUN_AT_UTC = {RUN_AT_UTC}')

RUN_AT_UTC = 2026-07-27T03:02:31.991666+00:00


## 1. 取得（6ページ）

サーバー負荷に配慮し、ページ間に短い待機（0.5秒）を入れる。

In [2]:
def parse_pub_text(raw_text):
    """“タイトル”\n著者\n掲載誌 の3行構造をパースする(改行数は1論文につき2箇所)。"""
    lines = [l.strip() for l in raw_text.split('\n') if l.strip()]
    title = lines[0].strip('“”"') if lines else None
    authors = lines[1] if len(lines) > 1 else None
    venue = lines[2] if len(lines) > 2 else None
    return title, authors, venue


all_rows = []
page_htmls = {}
for n in range(1, N_PAGES + 1):
    url = BASE_URL.format(n=n)
    resp = requests.get(url, headers={'User-Agent': USER_AGENT, 'Accept': 'text/html'}, timeout=30)
    resp.raise_for_status()
    # HTTPヘッダーにcharsetがなくrequestsがISO-8859-1と誤判定するため、明示的にUTF-8を指定する
    resp.encoding = 'utf-8'
    page_htmls[n] = resp.text
    soup = BeautifulSoup(resp.text, 'html.parser')
    items = soup.select('article.item.d_flex')
    print(f'page{n}: HTTP {resp.status_code}, items={len(items)}')

    for it in items:
        cat_el = it.select_one('.cat')
        category = cat_el.get_text(strip=True) if cat_el else None
        txt_el = it.select_one('.txt')
        title, authors, venue = parse_pub_text(txt_el.get_text('\n')) if txt_el else (None, None, None)
        link_el = it.select_one('a.link')
        link = link_el.get('href') if link_el else None

        all_rows.append({
            'page': n,
            'category': category,
            'title': title,
            'authors_raw': authors,
            'venue': venue,
            'link': link,
        })
    time.sleep(0.5)

df = pd.DataFrame(all_rows)
print(f'合計行数: {len(df)}')
df.head(5)

page1: HTTP 200, items=17


page2: HTTP 200, items=17


page3: HTTP 200, items=17


page4: HTTP 200, items=17


page5: HTTP 200, items=17


page6: HTTP 200, items=12


合計行数: 97


,page,category,title,authors_raw,venue,link
0,1,センシング,Exploiting broad incident angle ranges in diff...,"Hironori Suzuki, Eslam Abubakr, Shiro Saito, T...","Proc. SPIE 13902, Silicon Photonics XXI, 13902...",https://doi.org/10.1117/12.3075489
1,1,超電導,Superconductivity in Ba5Ir7Ge4 derived from Mg...,"Terunari Koshinuma, Hiroshi Fujihisa, Yoshito ...","Intermetallics, Volume 181, 2025, 108748",https://doi.org/10.1016/j.intermet.2025.108748
2,1,超電導,100 K級の転移温度を有するカルシウムフリー二層構造銅酸化物超伝導体の高圧合成性,"川島 健司, 二宮 博樹, 藤久 裕司, 伊豫 彰","高圧力の科学と技術, 2025 年 35 巻 2 号 p. 85-92",https://doi.org/10.4131/jshpreview.35.85
3,1,超電導,Rapid and scalable synthesis of alkali metal-i...,"A. Iyo, H. Fujihisa, Y. Gotoh, S. Ishida, H. E...","Supercond. Sci. Technol., 38, 2025, 055013",https://doi.org/10.1088/1361-6668/adcc63
4,1,超電導,Superconductivity of Sc20BC27 at ambient and u...,"G. Lamura, H. Ninomiya, R. Khasanov, K. Kawash...","Phys. Rev. B, 111, 2025, 214513",https://doi.org/10.1103/3w8q-c26h


## 2. 保存

In [3]:
raw_dir = OUTPUT_DIR / 'aisin_imra_jp_raw_pages'
raw_dir.mkdir(exist_ok=True)
for n, html in page_htmls.items():
    (raw_dir / f'page{n}.html').write_text(html, encoding='utf-8')

csv_path = OUTPUT_DIR / 'aisin_imra_jp.csv'
df.to_csv(csv_path, index=False)

meta_path = OUTPUT_DIR / 'aisin_imra_jp_meta.json'
with meta_path.open('w', encoding='utf-8') as f:
    json.dump({'run_at_utc': RUN_AT_UTC, 'source_url_pattern': BASE_URL, 'n_pages': N_PAGES, 'n_items': len(df)}, f, ensure_ascii=False, indent=2)

print(f'保存: {csv_path.relative_to(ROOT)}')
print(f'保存: {meta_path.relative_to(ROOT)}')
print(f'保存: {raw_dir.relative_to(ROOT)}/ (6ページ分の生HTML)')

保存: data/derived/toyota_official_scrape/aisin_imra_jp.csv
保存: data/derived/toyota_official_scrape/aisin_imra_jp_meta.json
保存: data/derived/toyota_official_scrape/aisin_imra_jp_raw_pages/ (6ページ分の生HTML)


## 3. ざっと確認

In [4]:
print('カテゴリ別件数:')
print(df['category'].value_counts())
print()
print('リンクがdoi.org以外/なし:', (~df['link'].fillna('').str.contains('doi.org')).sum(), '/', len(df))
print('著者欄が空:', df['authors_raw'].isna().sum())

カテゴリ別件数:
category
超電導         37
ライフサイエンス    21
太陽電池        18
センシング       16
燃料電池         5
Name: count, dtype: int64

リンクがdoi.org以外/なし: 23 / 97
著者欄が空: 1
